# Bong Lem Final Campaign Report

This notebook builds a full end-of-campaign business report from:

- Traffic exports in `total-data/`
- Order data in `bonglem.orders.json`

Outputs:
- KPI dashboards and trend charts
- Audience and channel breakdowns
- A generated markdown report at `report/final_campaign_report.md`

In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "total-data"
ORDERS_PATH = BASE_DIR / "bonglem.orders.json"
REPORT_PATH = BASE_DIR / "report" / "final_campaign_report.md"


def read_csv_with_fallback(path: Path) -> pd.DataFrame:
    last_error = None
    for encoding in ("utf-8-sig", "utf-8", "cp1258", "cp1252"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise ValueError(f"Could not decode {path}") from last_error


def find_one_file(data_dir: Path, pattern: str) -> Path:
    matches = sorted(data_dir.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No files matching pattern {pattern!r} in {data_dir}")
    return matches[0]


def parse_date_labels(labels: pd.Series, base_year: int) -> pd.Series:
    parsed_dates: list[pd.Timestamp | pd.NaT] = []
    year = base_year
    previous_month = None

    for value in labels.astype(str):
        text = value.strip()
        if not text or text.lower() == "nan":
            parsed_dates.append(pd.NaT)
            continue

        try:
            partial = datetime.strptime(f"{text} 2000", "%b %d %Y")
        except ValueError:
            parsed_dates.append(pd.NaT)
            continue

        if previous_month is not None and partial.month < previous_month:
            year += 1
        previous_month = partial.month

        parsed_dates.append(pd.Timestamp(year=year, month=partial.month, day=partial.day))

    return pd.Series(parsed_dates)


def load_daily_traffic(data_dir: Path) -> pd.DataFrame:
    visitor_file = find_one_file(data_dir, "*Visitor_viewer*.csv")
    base_year = datetime.fromtimestamp(visitor_file.stat().st_mtime).year

    raw = read_csv_with_fallback(visitor_file).iloc[:, :4].copy()
    raw.columns = ["date_label", "helper_value", "visitors_named", "page_views"]

    helper = pd.to_numeric(raw["helper_value"], errors="coerce")
    visitors_named = pd.to_numeric(raw["visitors_named"], errors="coerce")
    page_views = pd.to_numeric(raw["page_views"], errors="coerce")

    # Some exported rows place visitors in the second numeric column.
    fallback_mask = visitors_named.isna() & helper.notna() & page_views.notna() & (helper <= page_views)
    visitors = visitors_named.copy()
    visitors.loc[fallback_mask] = helper.loc[fallback_mask]

    daily = pd.DataFrame(
        {
            "date": parse_date_labels(raw["date_label"], base_year=base_year),
            "visitors": visitors,
            "page_views": page_views,
        }
    )

    daily = daily.loc[~daily[["visitors", "page_views"]].isna().all(axis=1)]
    daily = daily.dropna(subset=["date"]).copy()
    daily["visitors"] = daily["visitors"].fillna(0).round().astype(int)
    daily["page_views"] = daily["page_views"].fillna(0).round().astype(int)

    daily = daily.groupby("date", as_index=False).agg(visitors=("visitors", "sum"), page_views=("page_views", "sum"))

    full_dates = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")
    daily = daily.set_index("date").reindex(full_dates, fill_value=0).rename_axis("date").reset_index()

    return daily


def load_segment_table(data_dir: Path, filename_keyword: str, segment_name: str) -> pd.DataFrame:
    path = find_one_file(data_dir, f"*{filename_keyword}*.csv")
    raw = read_csv_with_fallback(path).iloc[:, :3].copy()
    raw.columns = [segment_name, "Visitors", "Total"]
    raw[segment_name] = raw[segment_name].astype(str).str.strip()
    raw["Visitors"] = pd.to_numeric(raw["Visitors"], errors="coerce").fillna(0)
    raw["Total"] = pd.to_numeric(raw["Total"], errors="coerce").fillna(0)
    raw = raw[raw[segment_name].ne("") & raw[segment_name].str.lower().ne("nan")]
    return raw.sort_values(["Total", "Visitors"], ascending=False).reset_index(drop=True)


def parse_mongo_date(value) -> pd.Timestamp | pd.NaT:
    if isinstance(value, dict) and "$date" in value:
        value = value["$date"]
    ts = pd.to_datetime(value, errors="coerce", utc=True)
    if pd.isna(ts):
        return pd.NaT
    return ts.tz_convert("Asia/Ho_Chi_Minh").tz_localize(None)


def load_orders(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8") as f:
        orders = json.load(f)

    df = pd.DataFrame(orders)
    if df.empty:
        return df

    df["created_at"] = df.get("createdAt", pd.Series(dtype=object)).apply(parse_mongo_date)
    df["updated_at"] = df.get("updatedAt", pd.Series(dtype=object)).apply(parse_mongo_date)
    df["status"] = df.get("status", pd.Series(dtype=object)).fillna("unknown").astype(str).str.lower()
    df["paymentMethod"] = df.get("paymentMethod", pd.Series(dtype=object)).fillna("unknown").astype(str).str.lower()
    df["paymentStatus"] = df.get("paymentStatus", pd.Series(dtype=object)).fillna("unknown").astype(str).str.lower()
    df["total"] = pd.to_numeric(df.get("total", 0), errors="coerce").fillna(0)
    df["subtotal"] = pd.to_numeric(df.get("subtotal", 0), errors="coerce").fillna(0)
    return df


def add_week_index(df: pd.DataFrame, date_col: str, campaign_start: pd.Timestamp) -> pd.DataFrame:
    out = df.copy()
    out["day_index"] = (out[date_col] - campaign_start).dt.days
    out["week_no"] = (out["day_index"] // 7) + 1
    return out

In [ ]:
daily = load_daily_traffic(DATA_DIR)

countries = load_segment_table(DATA_DIR, "Country", "Country")
devices = load_segment_table(DATA_DIR, "Device", "Device")
os_table = load_segment_table(DATA_DIR, "Operating System", "OperatingSystem")
referrals = load_segment_table(DATA_DIR, "Referral", "Referrer")
orders = load_orders(ORDERS_PATH)

campaign_start = daily["date"].min()
campaign_end = daily["date"].max()

orders_campaign = orders[(orders["created_at"] >= campaign_start) & (orders["created_at"] <= campaign_end + pd.Timedelta(days=1))].copy()
daily = add_week_index(daily, "date", campaign_start)
orders_campaign = add_week_index(orders_campaign, "created_at", campaign_start)

print(f"Campaign period: {campaign_start.date()} -> {campaign_end.date()} ({len(daily)} days)")
print(f"Rows loaded | daily traffic: {len(daily)} | orders (all): {len(orders)} | orders (campaign period): {len(orders_campaign)}")

display(daily.head())
display(orders_campaign[["created_at", "status", "paymentMethod", "paymentStatus", "total"]].head())

In [ ]:
traffic_kpis = {
    "Visitors": int(daily["visitors"].sum()),
    "Page Views": int(daily["page_views"].sum()),
    "Views / Visitor": (daily["page_views"].sum() / daily["visitors"].sum()) if daily["visitors"].sum() else 0,
    "Avg Daily Visitors": daily["visitors"].mean(),
    "Peak Visitor Day": daily.loc[daily["visitors"].idxmax(), "date"].date().isoformat(),
    "Peak Visitors": int(daily["visitors"].max()),
}

completed_mask = orders_campaign["status"].eq("completed")
paid_mask = orders_campaign["paymentStatus"].eq("paid")

order_kpis = {
    "Orders (Campaign Period)": int(len(orders_campaign)),
    "Completed Orders": int(completed_mask.sum()),
    "Completion Rate": (completed_mask.mean() if len(orders_campaign) else 0),
    "Paid Orders": int(paid_mask.sum()),
    "Paid Rate": (paid_mask.mean() if len(orders_campaign) else 0),
    "Total GMV (All Orders)": float(orders_campaign["total"].sum()),
    "Completed GMV": float(orders_campaign.loc[completed_mask, "total"].sum()),
    "AOV (Completed)": float(orders_campaign.loc[completed_mask, "total"].mean() if completed_mask.any() else 0),
    "Est. Visitor->Completed Order Conversion": (completed_mask.sum() / daily["visitors"].sum()) if daily["visitors"].sum() else 0,
}

kpi_df = pd.DataFrame(
    {
        "Metric": list(traffic_kpis.keys()) + list(order_kpis.keys()),
        "Value": list(traffic_kpis.values()) + list(order_kpis.values()),
    }
)

weekly_traffic = daily.groupby("week_no", as_index=False).agg(
    Visitors=("visitors", "sum"),
    PageViews=("page_views", "sum"),
)
weekly_traffic["ViewsPerVisitor"] = np.where(
    weekly_traffic["Visitors"] > 0,
    weekly_traffic["PageViews"] / weekly_traffic["Visitors"],
    0,
)

weekly_orders = orders_campaign.groupby("week_no", as_index=False).agg(
    Orders=("status", "size"),
    CompletedOrders=("status", lambda s: (s == "completed").sum()),
    PaidOrders=("paymentStatus", lambda s: (s == "paid").sum()),
    GMV=("total", "sum"),
)

weekly = weekly_traffic.merge(weekly_orders, on="week_no", how="left").fillna(0)
weekly["CompletedRate"] = np.where(weekly["Orders"] > 0, weekly["CompletedOrders"] / weekly["Orders"], 0)
weekly["EstConversion"] = np.where(weekly["Visitors"] > 0, weekly["CompletedOrders"] / weekly["Visitors"], 0)

weekly["WoW Visitors %"] = weekly["Visitors"].pct_change().replace([np.inf, -np.inf], np.nan)
weekly["WoW Completed Orders %"] = weekly["CompletedOrders"].pct_change().replace([np.inf, -np.inf], np.nan)

print("Executive KPI Table")
display(kpi_df)

print("Weekly Performance")
display(weekly.round(4))

In [ ]:
top_countries = countries.head(10).copy()
top_devices = devices.head(10).copy()
top_os = os_table.head(10).copy()
top_referrals = referrals.head(10).copy()

status_table = (
    orders_campaign.groupby("status", as_index=False)
    .agg(Orders=("status", "size"), GMV=("total", "sum"))
    .sort_values("Orders", ascending=False)
)

payment_method_table = (
    orders_campaign.groupby("paymentMethod", as_index=False)
    .agg(Orders=("paymentMethod", "size"), GMV=("total", "sum"))
    .sort_values("Orders", ascending=False)
)

print("Top Countries")
display(top_countries)

print("Top Devices")
display(top_devices)

print("Top Operating Systems")
display(top_os)

print("Top Referrals")
display(top_referrals)

print("Order Status Performance")
display(status_table)

print("Payment Method Performance")
display(payment_method_table)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Daily traffic trend
ax = axes[0, 0]
ax.plot(daily["date"], daily["visitors"], label="Visitors", linewidth=2)
ax.plot(daily["date"], daily["page_views"], label="Page Views", linewidth=2, alpha=0.8)
ax.plot(daily["date"], daily["visitors"].rolling(7, min_periods=1).mean(), label="Visitors 7D Avg", linestyle="--")
ax.set_title("Daily Traffic Trend")
ax.set_xlabel("Date")
ax.set_ylabel("Count")
ax.legend()
ax.grid(alpha=0.3)

# Weekly conversion trend
ax = axes[0, 1]
ax.bar(weekly["week_no"] - 0.2, weekly["Visitors"], width=0.4, label="Visitors")
ax.bar(weekly["week_no"] + 0.2, weekly["CompletedOrders"], width=0.4, label="Completed Orders")
ax2 = ax.twinx()
ax2.plot(weekly["week_no"], weekly["EstConversion"], color="black", marker="o", label="Est Conversion")
ax.set_title("Weekly Visitors vs Completed Orders")
ax.set_xlabel("Week Number")
ax.set_ylabel("Volume")
ax2.set_ylabel("Conversion")
ax.grid(alpha=0.3)

# Top countries by visitors
ax = axes[1, 0]
country_plot = top_countries.sort_values("Visitors", ascending=True)
ax.barh(country_plot["Country"], country_plot["Visitors"])
ax.set_title("Top Countries by Visitors")
ax.set_xlabel("Visitors")

# Order status distribution
ax = axes[1, 1]
ax.bar(status_table["status"], status_table["Orders"])
ax.set_title("Order Status Distribution")
ax.set_xlabel("Status")
ax.set_ylabel("Orders")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def format_int(x):
    return f"{int(x):,}"


def format_currency(x):
    return f"{x:,.0f}"


def format_pct(x, digits=1):
    return f"{100 * x:.{digits}f}%"


def md_table(df: pd.DataFrame) -> str:
    headers = list(df.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for _, row in df.iterrows():
        values = [str(v).replace("|", "\\|") for v in row.values]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


traffic_total = daily["visitors"].sum()
completed_orders = int(order_kpis["Completed Orders"])

best_week = weekly.sort_values("Visitors", ascending=False).head(1)
weakest_week = weekly.sort_values("Visitors", ascending=True).head(1)

country_share = (top_countries.iloc[0]["Visitors"] / countries["Visitors"].sum()) if len(countries) else 0
device_share = (
    devices.loc[devices["Device"].str.lower().eq("mobile"), "Visitors"].sum() / devices["Visitors"].sum()
    if len(devices)
    else 0
)

insights = []
insights.append(
    f"Traffic reached {format_int(traffic_total)} visitors and {format_int(daily['page_views'].sum())} page views during the campaign window."
)
insights.append(
    f"The campaign converted approximately {format_pct(order_kpis['Est. Visitor->Completed Order Conversion'], 2)} of visitors into completed orders ({format_int(completed_orders)} orders)."
)
if order_kpis["Paid Rate"] < 0.5:
    insights.append(
        f"Payment collection is a critical gap: only {format_pct(order_kpis['Paid Rate'], 1)} of campaign-period orders are marked paid."
    )
if len(best_week):
    insights.append(
        f"Best traffic week was Week {int(best_week.iloc[0]['week_no'])} with {format_int(best_week.iloc[0]['Visitors'])} visitors."
    )
if len(weakest_week):
    insights.append(
        f"Lowest traffic week was Week {int(weakest_week.iloc[0]['week_no'])} with {format_int(weakest_week.iloc[0]['Visitors'])} visitors."
    )
if country_share > 0.5:
    insights.append(
        f"Audience concentration risk exists: top country contributes {format_pct(country_share, 1)} of visitors."
    )
if device_share > 0.5:
    insights.append(
        f"Mobile-first execution is mandatory because mobile contributes {format_pct(device_share, 1)} of visitors."
    )

weekly_report_df = weekly.copy()
weekly_report_df["ViewsPerVisitor"] = weekly_report_df["ViewsPerVisitor"].map(lambda x: f"{x:.2f}")
weekly_report_df["CompletedRate"] = weekly_report_df["CompletedRate"].map(lambda x: format_pct(x, 1))
weekly_report_df["EstConversion"] = weekly_report_df["EstConversion"].map(lambda x: format_pct(x, 2))
weekly_report_df["GMV"] = weekly_report_df["GMV"].map(format_currency)

summary_kpi_df = pd.DataFrame(
    [
        ["Campaign period", f"{campaign_start.date()} to {campaign_end.date()}"],
        ["Total visitors", format_int(traffic_kpis["Visitors"])],
        ["Total page views", format_int(traffic_kpis["Page Views"])],
        ["Views per visitor", f"{traffic_kpis['Views / Visitor']:.2f}"],
        ["Campaign orders", format_int(order_kpis["Orders (Campaign Period)"])],
        ["Completed orders", format_int(order_kpis["Completed Orders"])],
        ["Completion rate", format_pct(order_kpis["Completion Rate"], 1)],
        ["Paid rate", format_pct(order_kpis["Paid Rate"], 1)],
        ["GMV (all orders)", format_currency(order_kpis["Total GMV (All Orders)"])],
        ["GMV (completed)", format_currency(order_kpis["Completed GMV"])],
        ["AOV (completed)", format_currency(order_kpis["AOV (Completed)"])],
    ],
    columns=["Metric", "Value"],
)

final_lines = [
    "# Bong Lem Campaign Final Report",
    "",
    f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    "## 1) Executive KPI Snapshot",
    "",
    md_table(summary_kpi_df),
    "",
    "## 2) Weekly Performance",
    "",
    md_table(weekly_report_df),
    "",
    "## 3) Audience & Acquisition Overview",
    "",
    "### Top Countries",
    "",
    md_table(top_countries[["Country", "Visitors", "Total"]]),
    "",
    "### Top Devices",
    "",
    md_table(top_devices[["Device", "Visitors", "Total"]]),
    "",
    "### Top Operating Systems",
    "",
    md_table(top_os[["OperatingSystem", "Visitors", "Total"]]),
    "",
    "### Top Referrals",
    "",
    md_table(top_referrals[["Referrer", "Visitors", "Total"]]),
    "",
    "## 4) Commercial Performance",
    "",
    "### Order Status",
    "",
    md_table(status_table),
    "",
    "### Payment Method",
    "",
    md_table(payment_method_table),
    "",
    "## 5) Key Business Insights",
    "",
]

for insight in insights:
    final_lines.append(f"- {insight}")

final_lines.extend(
    [
        "",
        "## 6) Recommended Actions for Next Campaign",
        "",
        "- Improve payment completion workflow with reminders and clearer payment instructions.",
        "- Build a weekly campaign cadence based on the strongest traffic week pattern in this report.",
        "- Prioritize mobile page speed, checkout UX, and above-the-fold product communication.",
        "- Add strict UTM standards for all social/referral links to improve attribution accuracy.",
        "- Track conversion funnel stages (visit -> add to cart -> checkout -> paid) for better optimization.",
    ]
)

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text("\n".join(final_lines) + "\n", encoding="utf-8")

print(f"Final report written to: {REPORT_PATH}")
print("\nPreview:\n")
print("\n".join(final_lines[:40]))

## Notes

- This report combines traffic and order data at campaign level, not user-level attribution.
- Use the markdown output in `report/final_campaign_report.md` as the executive deliverable.
- Re-run all cells whenever `total-data/` or `bonglem.orders.json` changes.